In [12]:
import os
import random
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import shap
import matplotlib.pyplot as plt
import cv2

# Configuration
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 8
BACKGROUND_SIZE = 50
EXPLAIN_SIZE = 40
MODEL_PATH = "outputs/models/best_densenet121_tb.pth"
TRAIN_CSV = "outputs/train_split.csv"
TEST_CSV = "outputs/test_split.csv"
OUTPUT_DIR = "outputs/shap"
HEATMAP_DIR = os.path.join(OUTPUT_DIR, "shap_heatmaps")
LOCAL_DIR = os.path.join(OUTPUT_DIR, "shap_local_examples")
CASE_DIR = os.path.join(OUTPUT_DIR, "case_examples")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(CASE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

Using device: cuda


In [16]:
# Dataset Class
class TBXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        label = int(row["label"])
        image_id = row["image_id"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = image
        return image_tensor, label, image_id, image_path

# Transform
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load Data and Select SHAP Sample
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# "balanced" = 30 Shenzhen + 30 Montgomery with equal TB/Non-TB where possible
# "montgomery_only" = only Montgomery images, for lung-mask relevance analysis
# SHAP_SAMPLE_MODE = "balanced"
SHAP_SAMPLE_MODE = "montgomery_only"

def add_dataset_name(df):
    """
    Add dataset_name column using image_path and image_id.
    """
    df = df.copy()

    path_lower = df["image_path"].astype(str).str.lower()
    id_lower = df["image_id"].astype(str).str.lower()

    df["dataset_name"] = np.select(
        [
            path_lower.str.contains("shenzhen") | id_lower.str.contains("chncxr"),
            path_lower.str.contains("montgomery") | id_lower.str.contains("mcucxr")
        ],
        [
            "Shenzhen",
            "Montgomery"
        ],
        default="Unknown"
    )

    return df

def make_balanced_dataset_sample(df, n_per_dataset=30, seed=42):
    """
    Select equal number of samples from Shenzhen and Montgomery.

    Target:
    - 30 Shenzhen total: 15 Non-TB + 15 TB
    - 30 Montgomery total: 15 Non-TB + 15 TB

    If a dataset/class group has fewer than required images, it uses all available.
    """
    df = add_dataset_name(df)

    samples = []
    n_per_class = n_per_dataset // 2

    print("\nAvailable samples before balancing:")
    print(df.groupby(["dataset_name", "label"]).size())

    for dataset_name in ["Shenzhen", "Montgomery"]:
        for label in [0, 1]:
            group_df = df[
                (df["dataset_name"] == dataset_name) &
                (df["label"] == label)
            ]

            available = len(group_df)
            n_select = min(n_per_class, available)

            if available < n_per_class:
                print(
                    f"Warning: only {available} images available for "
                    f"{dataset_name}, label {label}. "
                    f"Requested {n_per_class}, using {n_select}."
                )

            if n_select > 0:
                sampled_group = group_df.sample(
                    n=n_select,
                    random_state=seed
                )
                samples.append(sampled_group)

    if len(samples) == 0:
        raise ValueError("No samples selected. Check dataset paths and labels.")

    balanced_df = pd.concat(samples, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    print("\nFinal balanced SHAP sample:")
    print(balanced_df.groupby(["dataset_name", "label"]).size())
    print("\nDataset totals:")
    print(balanced_df["dataset_name"].value_counts())
    print("\nClass totals:")
    print(balanced_df["label"].value_counts())

    return balanced_df

def make_montgomery_only_sample(train_df, test_df, use_all_splits=True, seed=42):
    """
    Select Montgomery-only images for lung-region relevance analysis.
    Uses train + val + test if val_split.csv exists.
    """
    if use_all_splits:
        val_path = "outputs/val_split.csv"

        if os.path.exists(val_path):
            val_df = pd.read_csv(val_path)
            all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
            print("Using train + val + test for Montgomery-only SHAP.")
        else:
            all_df = pd.concat([train_df, test_df], ignore_index=True)
            print("val_split.csv not found. Using train + test only.")
    else:
        all_df = test_df.copy()
        print("Using test split only for Montgomery-only SHAP.")

    all_df = add_dataset_name(all_df)

    montgomery_df = all_df[
        all_df["dataset_name"] == "Montgomery"
    ].drop_duplicates(subset=["image_id"]).reset_index(drop=True)

    montgomery_df = montgomery_df.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    print("\nMontgomery-only SHAP sample:")
    print("Total Montgomery images:", len(montgomery_df))
    print("\nClass distribution:")
    print(montgomery_df["label"].value_counts())

    if len(montgomery_df) == 0:
        raise ValueError(
            "No Montgomery images found. Check image_path or image_id naming."
        )

    return montgomery_df

# Select output folders and SHAP dataframe
if SHAP_SAMPLE_MODE == "balanced":
    OUTPUT_DIR = "outputs/shap_balanced"
    selected_test_df = make_balanced_dataset_sample(
        test_df,
        n_per_dataset=30,
        seed=SEED
    )

elif SHAP_SAMPLE_MODE == "montgomery_only":
    OUTPUT_DIR = "outputs/shap_montgomery"
    selected_test_df = make_montgomery_only_sample(
        train_df,
        test_df,
        use_all_splits=True,
        seed=SEED
    )

else:
    raise ValueError(f"Unknown SHAP_SAMPLE_MODE: {SHAP_SAMPLE_MODE}")

# Recreate output folders based on selected mode
HEATMAP_DIR = os.path.join(OUTPUT_DIR, "shap_heatmaps")
LOCAL_DIR = os.path.join(OUTPUT_DIR, "shap_local_examples")
CASE_DIR = os.path.join(OUTPUT_DIR, "case_examples")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(CASE_DIR, exist_ok=True)

# Create Datasets and DataLoaders
train_dataset = TBXrayDataset(train_df, transform=transform)
test_dataset = TBXrayDataset(selected_test_df, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("\nSelected SHAP dataframe preview:")
print(selected_test_df[["image_id", "dataset_name", "label", "image_path"]].head())
print("\nSelected SHAP sample size:", len(selected_test_df))

# Load DenseNet121 Model
model = models.densenet121(weights=None)
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Wrapper for SHAP
class TBLogitModel(nn.Module):
    """
    Returns only the TB class logit.
    SHAP is applied to the TB logit instead of softmax probability because
    gradients through logits are usually more stable than gradients through
    saturated softmax probabilities.
    """
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        outputs = self.base_model(x)
        return outputs[:, 1:2]


wrapped_model = TBLogitModel(model).to(DEVICE)
wrapped_model.eval()

# Helper Functions
def denormalize_image(tensor):
    """
    Convert normalised torch tensor back to displayable RGB image.
    """
    image = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image, 0, 1)
    return image


def shap_to_heatmap(shap_value):
    """
    Convert SHAP values to a normalised 2D heatmap.

    Expected input:
    - C x H x W
    """
    shap_value = np.asarray(shap_value)
    if shap_value.ndim != 3:
        raise ValueError(f"Expected SHAP value with 3 dimensions, got shape {shap_value.shape}")
    if shap_value.shape[0] == 3:
        shap_abs = np.abs(shap_value).mean(axis=0)
    elif shap_value.shape[-1] == 3:
        shap_abs = np.abs(shap_value).mean(axis=-1)
    else:
        raise ValueError(f"Unexpected SHAP value shape: {shap_value.shape}")
    shap_abs = shap_abs - shap_abs.min()
    shap_abs = shap_abs / (shap_abs.max() + 1e-8)
    return shap_abs


def save_shap_overlay(original_image, heatmap, save_path, title):
    """
    Save original image with SHAP heatmap overlay.
    """
    heatmap_resized = cv2.resize(
        heatmap,
        (original_image.shape[1], original_image.shape[0])
    )
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.55 * original_image + 0.45 * heatmap_color
    overlay = np.clip(overlay, 0, 1)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def save_original_heatmap_overlay(original_image, heatmap, save_path, title):
    """
    Save a 3-panel figure:
    original image, SHAP heatmap, overlay.
    """
    heatmap_resized = cv2.resize(
        heatmap,
        (original_image.shape[1], original_image.shape[0])
    )

    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.55 * original_image + 0.45 * heatmap_color
    overlay = np.clip(overlay, 0, 1)

    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(original_image)
    plt.axis("off")
    plt.title("Original X-ray")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="hot")
    plt.axis("off")
    plt.title("SHAP Heatmap")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("Overlay")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def normalise_shap_return(shap_values):
    """
    Handles SHAP version-dependent return formats.

    Desired final shape:
    N x C x H x W
    """
    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.asarray(shap_values)
    # Sometimes output is N x C x H x W x 1
    if shap_values.ndim == 5 and shap_values.shape[-1] == 1:
        shap_values = shap_values[..., 0]

    # Sometimes output is N x H x W x C
    if shap_values.ndim == 4 and shap_values.shape[-1] == 3:
        shap_values = np.transpose(shap_values, (0, 3, 1, 2))

    if shap_values.ndim != 4:
        raise ValueError(f"Unexpected final SHAP shape: {shap_values.shape}")

    if shap_values.shape[1] != 3:
        raise ValueError(f"Expected channel dimension at axis 1, got shape: {shap_values.shape}")
        
    return shap_values


def get_model_predictions(model, images):
    """
    Return CNN predicted labels and TB probabilities.
    """
    model.eval()
    with torch.no_grad():
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        tb_probabilities = probabilities[:, 1]
        predictions = torch.argmax(probabilities, dim=1)

    return predictions.cpu().numpy(), tb_probabilities.cpu().numpy()

Using train + val + test for Montgomery-only SHAP.

Montgomery-only SHAP sample:
Total Montgomery images: 138

Class distribution:
label
0    80
1    58
Name: count, dtype: int64

Selected SHAP dataframe preview:
        image_id dataset_name  label  \
0  MCUCXR_0255_1   Montgomery      1   
1  MCUCXR_0309_1   Montgomery      1   
2  MCUCXR_0173_1   Montgomery      1   
3  MCUCXR_0059_0   Montgomery      0   
4  MCUCXR_0150_1   Montgomery      1   

                                          image_path  
0  /user/HS401/mn01409/Documents/Dissertation/dat...  
1  /user/HS401/mn01409/Documents/Dissertation/dat...  
2  /user/HS401/mn01409/Documents/Dissertation/dat...  
3  /user/HS401/mn01409/Documents/Dissertation/dat...  
4  /user/HS401/mn01409/Documents/Dissertation/dat...  

Selected SHAP sample size: 138


In [17]:
# Select Background Samples
background_images = []

for images, labels, image_ids, image_paths in train_loader:
    background_images.append(images)

    if len(background_images) * BATCH_SIZE >= BACKGROUND_SIZE:
        break

background = torch.cat(background_images, dim=0)[:BACKGROUND_SIZE].to(DEVICE)

print("Background shape:", background.shape)

# Select Test Samples to Explain
test_images = []
test_labels = []
test_ids = []
test_paths = []
test_dataset_names = []

for images, labels, image_ids, image_paths in test_loader:
    test_images.append(images)
    test_labels.extend(labels.numpy())
    test_ids.extend(image_ids)
    test_paths.extend(image_paths)

test_images = torch.cat(test_images, dim=0).to(DEVICE)
test_labels = np.array(test_labels)
test_ids = list(test_ids)
test_paths = list(test_paths)

# Recover dataset names from selected_test_df in the same order
test_dataset_names = selected_test_df["dataset_name"].tolist()

print("Test explanation shape:", test_images.shape)
print("Number of images selected for SHAP:", len(test_ids))
print(pd.Series(test_dataset_names).value_counts())

# Get DenseNet Predictions
cnn_predictions, cnn_probabilities = get_model_predictions(model, test_images)

# Run SHAP GradientExplainer
print("Creating SHAP GradientExplainer...")
explainer = shap.GradientExplainer(
    wrapped_model,
    background
)
print("Generating SHAP values...")
shap_values = explainer.shap_values(
    test_images,
    nsamples=200,
    rseed=SEED
)

shap_values = normalise_shap_return(shap_values)
print("Final SHAP values shape:", shap_values.shape)

# Save SHAP Values and Metadata
np.save(os.path.join(OUTPUT_DIR, "shap_values.npy"), shap_values)

meta_df = pd.DataFrame({
    "image_id": test_ids,
    "image_path": test_paths,
    "dataset_name": test_dataset_names,
    "true_label": test_labels,
    "cnn_prediction": cnn_predictions,
    "cnn_probability_tb": cnn_probabilities
})

meta_df["case_type"] = np.select(
    [
        (meta_df["true_label"] == 1) & (meta_df["cnn_prediction"] == 1),
        (meta_df["true_label"] == 0) & (meta_df["cnn_prediction"] == 0),
        (meta_df["true_label"] == 0) & (meta_df["cnn_prediction"] == 1),
        (meta_df["true_label"] == 1) & (meta_df["cnn_prediction"] == 0)
    ],
    ["TP","TN","FP","FN"],
    default="Unknown"
)

meta_df.to_csv(os.path.join(OUTPUT_DIR, "shap_metadata.csv"), index=False)
print("\nSaved SHAP metadata:")
print(meta_df.groupby(["dataset_name", "true_label", "case_type"]).size())

settings = {
    "seed": SEED,
    "image_size": IMG_SIZE,
    "background_size": BACKGROUND_SIZE,
    "explain_size": EXPLAIN_SIZE,
    "model_path": MODEL_PATH,
    "train_csv": TRAIN_CSV,
    "test_csv": TEST_CSV,
    "explainer": "shap.GradientExplainer",
    "explained_output": "TB logit"
}

with open(os.path.join(OUTPUT_DIR, "shap_settings.json"), "w") as f:
    json.dump(settings, f, indent=4)
    
# Save Local SHAP Heatmaps
test_images_cpu = test_images.detach().cpu()
for i in range(len(test_ids)):
    original = denormalize_image(test_images_cpu[i])
    heatmap = shap_to_heatmap(shap_values[i])
    title = (
        f"SHAP | ID: {test_ids[i]} | "
        f"True: {test_labels[i]} | "
        f"CNN Pred: {cnn_predictions[i]} | "
        f"TB Prob: {cnn_probabilities[i]:.3f}"
    )
    overlay_path = os.path.join(
        HEATMAP_DIR,
        f"{test_ids[i]}_shap_overlay.png"
    )
    panel_path = os.path.join(
        LOCAL_DIR,
        f"{test_ids[i]}_shap_panel.png"
    )
    save_shap_overlay(
        original,
        heatmap,
        overlay_path,
        title
    )
    save_original_heatmap_overlay(
        original,
        heatmap,
        panel_path,
        title
    )
print("Saved local SHAP heatmaps and panels.")

# Save TP / TN / FP / FN Examples
for case_type in ["TP", "TN", "FP", "FN"]:
    case_rows = meta_df[meta_df["case_type"] == case_type]
    if len(case_rows) == 0:
        print(f"No {case_type} example found in explained sample.")
        continue
    idx = case_rows.index[0]
    original = denormalize_image(test_images_cpu[idx])
    heatmap = shap_to_heatmap(shap_values[idx])
    title = (
        f"{case_type} Example | ID: {test_ids[idx]} | "
        f"True: {test_labels[idx]} | "
        f"CNN Pred: {cnn_predictions[idx]} | "
        f"TB Prob: {cnn_probabilities[idx]:.3f}"
    )
    save_path = os.path.join(
        CASE_DIR,
        f"{case_type}_shap_example.png"
    )
    save_original_heatmap_overlay(
        original,
        heatmap,
        save_path,
        title
    )
print("Saved TP/TN/FP/FN examples where available.")

# Create Global SHAP Importance Map
global_shap = np.mean(np.abs(shap_values), axis=0)
global_heatmap = shap_to_heatmap(global_shap)

plt.figure(figsize=(6, 6))
plt.imshow(global_heatmap, cmap="hot")
plt.axis("off")
plt.title(f"Global SHAP Importance Map, n={len(test_ids)}")
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "shap_global_importance_map.png"),
    dpi=300
)
plt.close()
np.save(os.path.join(OUTPUT_DIR, "shap_global_importance.npy"), global_heatmap)
print("Saved global SHAP importance map.")

# Save Case Summary Counts
case_summary = meta_df["case_type"].value_counts().reset_index()
case_summary.columns = ["case_type", "count"]
case_summary.to_csv(os.path.join(OUTPUT_DIR, "shap_case_summary.csv"), index=False)
print("\nCase summary:")
print(case_summary)

Background shape: torch.Size([50, 3, 224, 224])
Test explanation shape: torch.Size([138, 3, 224, 224])
Number of images selected for SHAP: 138
Montgomery    138
Name: count, dtype: int64
Creating SHAP GradientExplainer...
Generating SHAP values...
Final SHAP values shape: (138, 3, 224, 224)

Saved SHAP metadata:
dataset_name  true_label  case_type
Montgomery    0           FP            3
                          TN           77
              1           TP           58
dtype: int64
Saved local SHAP heatmaps and panels.
No FN example found in explained sample.
Saved TP/TN/FP/FN examples where available.
Saved global SHAP importance map.

Case summary:
  case_type  count
0        TN     77
1        TP     58
2        FP      3


In [18]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2

# Configuration
IMG_SIZE = 224
SHAP_DIR = "outputs/shap"
SHAP_VALUES_PATH = os.path.join(SHAP_DIR, "shap_values.npy")
METADATA_PATH = os.path.join(SHAP_DIR, "shap_metadata.csv")
OUTPUT_DIR = os.path.join(SHAP_DIR, "visual_summary")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load SHAP values and metadata
shap_values = np.load(SHAP_VALUES_PATH)
meta_df = pd.read_csv(METADATA_PATH)
print("SHAP values shape:", shap_values.shape)
print(meta_df.head())

# Transform for loading original image
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def denormalize_image(tensor):
    image = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image, 0, 1)
    return image

def load_processed_image(image_path):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image)
    return denormalize_image(tensor)

def shap_to_heatmap(shap_value):
    shap_value = np.asarray(shap_value)

    if shap_value.ndim != 3:
        raise ValueError(f"Expected 3D SHAP value, got {shap_value.shape}")

    if shap_value.shape[0] == 3:
        shap_abs = np.abs(shap_value).mean(axis=0)
    elif shap_value.shape[-1] == 3:
        shap_abs = np.abs(shap_value).mean(axis=-1)
    else:
        raise ValueError(f"Unexpected SHAP shape: {shap_value.shape}")
    shap_abs = shap_abs - shap_abs.min()
    shap_abs = shap_abs / (shap_abs.max() + 1e-8)
    return shap_abs

def create_overlay(original_image, heatmap):
    heatmap_resized = cv2.resize(
        heatmap,
        (original_image.shape[1], original_image.shape[0])
    )
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.55 * original_image + 0.45 * heatmap_color
    overlay = np.clip(overlay, 0, 1)
    return heatmap_resized, overlay

def plot_case_examples(meta_df, shap_values, save_path):
    selected_indices = []
    for case_type in ["TP", "TN", "FP", "FN"]:
        case_rows = meta_df[meta_df["case_type"] == case_type]
        if len(case_rows) > 0:
            selected_indices.append(case_rows.index[0])
    if len(selected_indices) == 0:
        raise ValueError("No TP/TN/FP/FN examples found.")
        
    n_rows = len(selected_indices)
    plt.figure(figsize=(12, 4 * n_rows))
    plot_idx = 1
    for idx in selected_indices:
        row = meta_df.iloc[idx]

        original = load_processed_image(row["image_path"])
        heatmap = shap_to_heatmap(shap_values[idx])
        heatmap_resized, overlay = create_overlay(original, heatmap)

        title = (
            f"{row['case_type']} | True: {row['true_label']} | "
            f"Pred: {row['cnn_prediction']} | "
            f"TB Prob: {row['cnn_probability_tb']:.3f}"
        )

        plt.subplot(n_rows, 3, plot_idx)
        plt.imshow(original)
        plt.axis("off")
        plt.title("Original")
        plot_idx += 1

        plt.subplot(n_rows, 3, plot_idx)
        plt.imshow(heatmap_resized, cmap="hot")
        plt.axis("off")
        plt.title("SHAP Heatmap")
        plot_idx += 1

        plt.subplot(n_rows, 3, plot_idx)
        plt.imshow(overlay)
        plt.axis("off")
        plt.title(title)
        plot_idx += 1

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
plot_case_examples(
    meta_df,
    shap_values,
    os.path.join(OUTPUT_DIR, "tp_tn_fp_fn_shap_summary.png")
)
print("Saved SHAP case summary figure.")

SHAP values shape: (40, 3, 224, 224)
        image_id                                         image_path  \
0  CHNCXR_0516_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
1  CHNCXR_0399_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
2  CHNCXR_0012_0  /user/HS401/mn01409/Documents/Dissertation/dat...   
3  CHNCXR_0530_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
4  CHNCXR_0601_1  /user/HS401/mn01409/Documents/Dissertation/dat...   

   true_label  cnn_prediction  cnn_probability_tb case_type  
0           1               1            0.999988        TP  
1           1               1            0.999902        TP  
2           0               0            0.154846        TN  
3           1               1            0.999918        TP  
4           1               1            0.954788        TP  
Saved SHAP case summary figure.
